# nDCG@5

전체 30개 질문을 읽고 dev 15개로 설정을 검토합니다.

## 실행 환경 준비

이 노트북은 제공 PDF에서 실제 텍스트를 추출합니다. 답변 생성 API는 호출하지 않습니다.

### 수식의 세 단계를 코드와 연결하기

검색 순위를 관련도 배열로 바꾸고, 위치 할인을 적용한 DCG를 계산한 뒤, 전체 정답을 가장 앞에 놓은 IDCG로 나누어 nDCG를 반환합니다. 손계산과 함수가 같다는 `assert`는 구현을 확인할 뿐 검색 품질을 높이지 않습니다. 이진 라벨과 등급 라벨은 같은 nDCG 이름을 써도 값의 의미가 다릅니다.

In [1]:
from pathlib import Path
import json, math, re
from collections import Counter
from pypdf import PdfReader

# 현재 폴더에서 저장소의 공개 RAG 자료를 찾습니다.
root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "labs/data/rag-corpus/manifest.json").exists())
corpus = root / "labs/data/rag-corpus"
dataset_path = root / "labs/part3/ch11-evaluation/data/evaluation-queries.jsonl"
dataset = [json.loads(line) for line in dataset_path.read_text(encoding="utf-8").splitlines()]
evaluation_items = [item for item in dataset if item["split"] == "dev"]
print(f"전체 질문: {len(dataset)}개")
print(f"설정 선택용 dev 질문: {len(evaluation_items)}개")

전체 질문: 30개
설정 선택용 dev 질문: 15개


## 공개 PDF를 페이지 단위 후보로 준비

스캔본은 OCR이 필요하므로 이번 기초 검색에서 제외합니다.

In [2]:
manifest = json.loads((corpus / "manifest.json").read_text(encoding="utf-8"))
pages = []
for item in manifest:
    if item["tier"] == "scanned":
        continue
    reader = PdfReader(corpus / item["file"])
    for page_no, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        if text.strip():
            pages.append({"doc_id": item["doc_id"], "page": page_no, "text": text})
print(f"검색 후보: {len(pages)}개 페이지")

검색 후보: 90개 페이지


## 같은 검색기를 모든 질문에 적용

단어와 한글 두 글자 조각의 겹침을 사용합니다. 점수가 같으면 문서 ID와 쪽 번호로 순서를 고정합니다.

In [3]:
def tokens(text):
    normalized = re.sub(r"\s+", " ", text.lower())
    words = re.findall(r"[가-힣a-z0-9]+", normalized)
    compact = re.sub(r"[^가-힣a-z0-9]", "", normalized)
    return words + [compact[i:i+2] for i in range(max(0, len(compact)-1))]

def search(question, limit=50):
    query_counts = Counter(tokens(question))
    scored = []
    for page in pages:
        page_counts = Counter(tokens(page["text"]))
        overlap = sum(min(count, page_counts[token]) for token, count in query_counts.items())
        score = overlap / math.sqrt(max(1, sum(page_counts.values())))
        scored.append({"doc_id": page["doc_id"], "page": page["page"], "score": score})
    ranked_pages = sorted(scored, key=lambda row: (-row["score"], row["doc_id"], row["page"]))
    unique_docs, seen = [], set()
    for row in ranked_pages:
        if row["doc_id"] not in seen:
            unique_docs.append(row)
            seen.add(row["doc_id"])
    return unique_docs[:limit]

sample = search(dataset[0]["question"], 5)
print([(row["doc_id"], row["page"], round(row["score"], 3)) for row in sample])

[('E01', 1, 0.926), ('M04', 3, 0.225), ('H03', 1, 0.188), ('H06', 5, 0.142), ('E02', 5, 0.132)]


## 예제 순위를 손으로 계산

관련도 1과 0만 사용하는 이진 라벨로 DCG와 이상적인 DCG를 비교합니다.

In [4]:
def ndcg_from_gains(gains):
    dcg = sum(gain / math.log2(rank + 2) for rank, gain in enumerate(gains))
    ideal = sorted(gains, reverse=True)
    idcg = sum(gain / math.log2(rank + 2) for rank, gain in enumerate(ideal))
    return dcg / idcg if idcg else 0.0

example = [0, 1, 0, 1, 0]
manual = (1 / math.log2(3) + 1 / math.log2(5)) / (1 + 1 / math.log2(3))
print("함수:", round(ndcg_from_gains(example), 6))
print("수계산:", round(manual, 6))
assert math.isclose(ndcg_from_gains(example), manual)

함수: 0.650921
수계산: 0.650921


## 개발용 15문항 nDCG@5

설정 선택에는 dev 분할만 사용합니다. 정답 문서가 앞에 있을수록 점수가 높습니다.

In [5]:
def ndcg_at(results, gold, k):
    gains = [1 if row["doc_id"] in set(gold) else 0 for row in results[:k]]
    dcg = sum(gain / math.log2(rank + 2) for rank, gain in enumerate(gains))
    idcg = sum(1 / math.log2(rank + 2) for rank in range(min(k, len(set(gold)))))
    return dcg / idcg if idcg else 0.0

scores = [ndcg_at(search(item["question"], 50), item["gold_doc_ids"], 5)
          for item in evaluation_items]
print("nDCG@5:", round(sum(scores) / len(scores), 3))
print("최솟값/최댓값:", round(min(scores), 3), round(max(scores), 3))

nDCG@5: 0.893
최솟값/최댓값: 0.5 1.0
